# Main.py

Ini buat Install Library

In [ ]:
!pip install --upgrade --force-reinstall langchain langchain-groq langchain-chroma langchain-community sentence-transformers==3.0.1 tokenizers==0.22.2 pydantic==2.8.2 protobuf==3.20.3

Ini buat import database

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

documents = [
    "Kebijakan cuti perusahaan: Karyawan berhak mendapatkan 10 hari cuti tahunan.",
    "Prosedur pengajuan lembur wajib melalui persetujuan manajer minimal 24 jam sebelumnya.",
    "Kontak darurat IT support ada di 0812-3456-7890 dengan nama Hyuga santa claus.",
    "Kelas UNPAM terbaik ada di 01TPLP002",
    "Nama mahasiswa di kelas 01TPLP002 adalah Mommy ita",
    "Ketua kelas di 01TPLP002 itu adalah Ratu Dyra Berliani",
    "Sang jenius di 01TPLP002 adalah Rexaldo Dhiya UlhaQ SANG JENIUS"
]

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = text_splitter.create_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

print("Database Dora the explorer berhasil dibuat yey >.<")

Yang ini,eksekusi buat RAG

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup API
os.environ["GROQ_API_KEY"] = "Rahasia gelap"

# 2. Panggil LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

# 3. Setup Prompt
system_prompt = (
    "Kamu adalah asisten AI yang sangat cerdas.\n"
    "Gunakan data berikut untuk menjawab pertanyaan karyawan.\n"
    "Jika jawabannya tidak ada di data ini, bilang saja 'Data tidak ditemukan'.\n\n"
    "Konteks:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 4. Fungsi buat ngerapiin teks dari database
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 5. RAG CHAIN MURNI (Tanpa langchain.chains!)
retriever = vectorstore.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Eksekusi
query = "Berapa jatah cuti tahunan karyawan?"
response = rag_chain.invoke(query)

print(f"Hasil AI: {response}")

Yang ini buat output pertanyaan sama hasil

In [ ]:
pertanyaan_baru = "Siapa rektor unpam"
output = rag_chain.invoke(pertanyaan_baru)

print(f"Pertanyaan: {pertanyaan_baru}")
print(f"Jawaban: {output}")